# Feature Enrichment Analysis of JHU-Transferred HF Clusters in All of Us

This notebook characterizes All of Us (AoU) heart failure clusters assigned by the transferred Johns Hopkins University (JHU) VaDE model. It combines patient-level features with cluster assignments and compares each cluster with the remaining cohort to identify distinguishing clinical characteristics.

## Required inputs

- `meta_HFCohort_deid.csv`: JHU covariate metadata.
- `feature_matrix_wide_deaths.csv`: Patient-level AoU feature matrix.
- `patient_cluster_assignments_jhu_transfer_deaths.csv`: Cluster assignments from JHU model transfer.
- `covariate_id_name_map.csv`: Mapping between covariate IDs and remapped feature names.
- `vade_trained_features.csv`: Feature names used to train the model.

## Analysis

Features are restricted to numeric covariates represented in the model's training feature list. Patient identifiers and model outputs are excluded from enrichment analysis.

For each feature, each cluster is compared with all other clusters combined:

- **Binary features:** Natural-log odds ratios are calculated with a 0.5 continuity correction applied to each contingency-table cell.
- **Continuous features:** Mean differences are standardized by the feature's standard deviation across the full cohort.
- **Descriptive statistics:** Within-cluster and overall cohort summaries include feature means and, for binary features, counts and percentages of patients with the feature.
- **Ranking:** Binary features are ranked within each cluster using the log odds ratio multiplied by the within-cluster prevalence percentage.

JHU covariate names are added to make the results interpretable.

## Output

The feature-level summaries and effect measures are saved to:

`Outputs/vade_assignments/cluster_enrichment_transfer_c4_deaths.tsv`

In [ ]:
library(dplyr)
library(tidyr)
library(tibble)

In [ ]:
# metadata for column reading
jhu_raw  <- read.csv('meta_HFCohort_deid.csv', header = TRUE)
jhu_cols <- as.numeric(jhu_raw[[1]])

jhu_df <- data.frame(
  covariate_id   = jhu_raw$covariateId,
  covariate_name = jhu_raw$covariateName,
  concept_id     = jhu_raw$conceptId,
  analysis_id    = jhu_raw$analysisId
)

head(jhu_df)

In [ ]:
# helper functions
is_binary_feature <- function(x) {
  u <- unique(na.omit(x))
  length(u) > 0 && all(u %in% c(0, 1))
}

compute_log_or <- function(data, feature, cluster_id) {
  v <- data[[feature]]; incl <- data$cluster == cluster_id
  a <- sum(v == 1 & incl,  na.rm = TRUE) + 0.5
  b <- sum(v == 0 & incl,  na.rm = TRUE) + 0.5
  c <- sum(v == 1 & !incl, na.rm = TRUE) + 0.5
  d <- sum(v == 0 & !incl, na.rm = TRUE) + 0.5
  log((a / b) / (c / d))
}

compute_smd <- function(data, feature, cluster_id) {
  v <- data[[feature]]; incl <- data$cluster == cluster_id
  s <- sd(v, na.rm = TRUE)
  if (is.na(s) || s == 0) return(0)
  (mean(v[incl], na.rm = TRUE) - mean(v[!incl], na.rm = TRUE)) / s
}

compute_effect <- function(data, feature, cluster_id) {
  if (is_binary_feature(data[[feature]])) compute_log_or(data, feature, cluster_id)
  else compute_smd(data, feature, cluster_id)
}

log_or <- function(a, b, c, d) {
  log(((a + 0.5) / (b + 0.5)) / ((c + 0.5) / (d + 0.5)))
}

# covariate name lookup from jhu_df
covariate_lookup <- jhu_df %>%
  select(covariate_id, covariate_name) %>%
  mutate(Feature = as.character(covariate_id))

In [ ]:
# load data
feature_matrix      <- read.csv("feature_matrix_wide_deaths.csv", check.names = FALSE)
cluster_assignments <- read.csv("patient_cluster_assignments_jhu_transfer_deaths.csv")

X_with_cluster <- feature_matrix %>%
  mutate(person_id = as.character(person_id)) %>%
  left_join(cluster_assignments %>% mutate(person_id = as.character(person_id)),
            by = "person_id") %>%
  rename(patient_id = person_id)

cat("X_with_cluster:", nrow(X_with_cluster), "patients,", ncol(X_with_cluster), "cols\n")
cat("Cluster distribution:\n"); print(table(X_with_cluster$cluster))

# load the id <-> name mapping and the trained feature list, then reverse-map to covariate_ids
id_name_map <- read.csv("covariate_id_name_map.csv", colClasses = "character")
trained_names <- read.csv("vade_trained_features.csv", colClasses = "character")$remapped_name

trained_covariate_ids <- id_name_map %>%
  filter(remapped_name %in% trained_names) %>%
  pull(covariate_id)

cat("Trained names:", length(unique(trained_names)),
    "| Matched covariate_ids:", length(trained_covariate_ids), "\n")

# enrichment analysis
features <- X_with_cluster %>%
  select(where(is.numeric)) %>%
  select(-cluster) %>%
  names()
features <- features[!grepl("(^|_)id$|patient_id|rowId", features)]

# restrict to exactly the features the model trained on,
# and exclude VaDE's own output columns (gamma_*, z_*, confidence)
features <- intersect(features, trained_covariate_ids)
features <- setdiff(features, c("confidence", grep("^gamma_|^z_", features, value = TRUE)))

cat("Features going into enrichment:", length(features), "\n")  # should be 238

ftype <- sapply(features, function(f)
  if (is_binary_feature(X_with_cluster[[f]])) "binary" else "continuous")

results <- expand.grid(cluster = unique(X_with_cluster$cluster),
                       Feature = features, stringsAsFactors = FALSE) %>%
  rowwise() %>%
  mutate(
    feature_type = ftype[[Feature]],
    Effect       = compute_effect(X_with_cluster, Feature, cluster),
    Log_OR       = ifelse(feature_type == "binary",     Effect, NA_real_),
    SMD          = ifelse(feature_type == "continuous", Effect, NA_real_),
    Cohens_d     = ifelse(feature_type == "binary", Effect * sqrt(3) / pi, Effect)
  ) %>%
  ungroup()

cluster_stat <- X_with_cluster %>%
  pivot_longer(all_of(features), names_to = "Feature", values_to = "val") %>%
  group_by(cluster, Feature) %>%
  summarise(cluster_n    = sum(val == 1, na.rm = TRUE),
            cluster_pct  = 100 * sum(val == 1, na.rm = TRUE) / n(),
            cluster_mean = mean(val, na.rm = TRUE), .groups = "drop")

cohort_stat <- X_with_cluster %>%
  pivot_longer(all_of(features), names_to = "Feature", values_to = "val") %>%
  group_by(Feature) %>%
  summarise(Cohort_n    = sum(val == 1, na.rm = TRUE),
            Cohort_pct  = 100 * sum(val == 1, na.rm = TRUE) / n(),
            cohort_mean = mean(val, na.rm = TRUE), .groups = "drop")

enrichment_results <- results %>%
  left_join(cluster_stat, by = c("cluster", "Feature")) %>%
  left_join(cohort_stat,  by = "Feature") %>%
  left_join(covariate_lookup, by = "Feature") %>%
  mutate(covariate_name = ifelse(is.na(covariate_name), Feature, covariate_name)) %>%
  mutate(weighted_score = Log_OR * cluster_pct) %>%
  arrange(cluster, desc(weighted_score)) %>%
  select(cluster, Feature, covariate_name, feature_type, Cohens_d, Effect, Log_OR, SMD,
         cluster_n, cluster_pct, cluster_mean, Cohort_n, Cohort_pct, cohort_mean, weighted_score)

write.table(enrichment_results,
            './Outputs/vade_assignments/cluster_enrichment_transfer_c4_deaths.tsv',
            quote = FALSE, row.names = FALSE, sep = '\t', na = '')

cat("Saved cluster_enrichment_transfer_c4_deaths.tsv\n")